## Import

In [ ]:

import numpy as np
import math
import matplotlib.pyplot as plt
import networkx as nx
from scipy import linalg
from inspect import getmembers
!pip install qiskit qiskit-aer
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import UnitaryGate
from scipy.linalg import expm
from qiskit import QuantumCircuit
from qiskit.quantum_info import random_unitary
from qiskit.compiler import transpile
from qiskit.quantum_info import state_fidelity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.1 MB/s eta 0:00:00


# Generazione Grafo calcolo autovettori e autovalori e creazione degli He

In [ ]:
def generateGraphWithNNodes(numberOfNodes):
  # Generazione di un grafo casuale
  n_nodes = numberOfNodes
  G = nx.erdos_renyi_graph(n_nodes, p=0.15, seed=42)

  # Calcolo della matrice Laplaciana
  L = nx.laplacian_matrix(G).toarray()

  print(f"Matrice Laplaciana ({n_nodes}x{n_nodes}) generata.")

  # Verifica validità (somma righe = 0)
  row_sums = np.sum(L, axis=1)
  # Si usa np.allclose per gestire la precisione dei floating point e controllare l'intero array
  if not np.allclose(row_sums, 0):
    print("Errore: La somma delle righe non è zero.")
    return None

  return L




In [ ]:
def generateSparseGraphWithNNodes(numberOfNodes):
    """
    Genera la matrice Laplaciana di un grafo connesso sparso dove il grado massimo
    è al più log2(numero di nodi) e il grado minimo è almeno 1.
    """
    import random
    d_max = int(math.log2(numberOfNodes))

    # Cerchiamo di generare un grafo che sia connesso e rispetti il vincolo
    # Usiamo un approccio iterativo per garantire la connessione
    connected = False
    attempts = 0
    while not connected and attempts < 100:
        # Probabilità bassa per mantenere il grafo sparso
        p = (d_max / numberOfNodes) * 0.8
        G = nx.fast_gnp_random_graph(numberOfNodes, p, seed=42+attempts)

        # Verifichiamo se il grado massimo è rispettato e se è connesso
        current_max_degree = max([d for n, d in G.degree()])
        if current_max_degree <= d_max and nx.is_connected(G):
            connected = True
        else:
            # Se non è connesso o grado troppo alto, proviamo a manipolarlo o rigenerarlo
            attempts += 1

    if not connected:
        # Fallback: crea un grafo a ciclo (grado 2) e aggiungi archi casuali
        G = nx.cycle_graph(numberOfNodes)
        nodes = list(G.nodes())
        for _ in range(numberOfNodes):
            u, v = random.sample(nodes, 2)
            if G.degree(u) < d_max and G.degree(v) < d_max:
                G.add_edge(u, v)

    L = nx.laplacian_matrix(G).toarray()
    actual_max = max([d for n, d in G.degree()])
    print(f"Grafo generato: N={numberOfNodes}, Grado Max={actual_max} (Limite={d_max}), Connesso={nx.is_connected(G)}")
    return L



def drawGraph(L, seed=42):
  # Ricostruiamo la matrice di adiacenza dalla Laplaciana
  # A = D - L. Poiché gli elementi fuori diagonale di L sono -1 se c'è un arco,
  # l'adiacenza ha 1 dove L ha valori negativi.
  adj = -L.copy()
  np.fill_diagonal(adj, 0)
  adj[adj > 0] = 1
  adj[adj < 0] = 0

  # Creazione del grafo da matrice di adiacenza
  G = nx.from_numpy_array(adj)

  plt.figure(figsize=(8, 6))
  pos = nx.spring_layout(G, seed=seed)

  nx.draw(G, pos, with_labels=True, node_color='skyblue',
          node_size=500, edge_color='gray', font_size=10, font_weight='bold')

  plt.title(f"Visualizzazione del Grafo dalla Matrice Laplaciana")


  plt.show()



def get_max_degree_from_laplacian(L):
    """
    Restituisce il grado massimo del grafo data la matrice Laplaciana L.
    """
    # I gradi dei nodi corrispondono agli elementi sulla diagonale della Laplaciana
    degrees = np.diag(L)
    return np.max(degrees)


def get_non_trivial_changed_states(matrix):
    """
    Returns the indices of the two basis states that are modified by a two-level unitary matrix.
    """
    matrix = np.array(matrix)
    n = matrix.shape[0]

    indices = []
    for i in range(n):
      for j in range(n):
        if matrix[i][j]== -1 :
          return i,j


###Costruzione contributo per ciasun arco

In [ ]:
def get_edge_contributions(L):
  """
  Data una matrice Laplaciana, restituisce una lista di matrici,
  una per ogni arco, che rappresentano il contributo di quell'arco alla Laplaciana totale.
  """
  n = L.shape[0]
  contributions = []

  # Iteriamo sulla parte triangolare superiore (esclusa la diagonale)
  # per trovare gli archi (dove L[i, j] < 0)
  for i in range(n):
    for j in range(i + 1, n):
      if L[i, j] < 0:
        # Trovato un arco tra i e j
        # La matrice di contributo per un arco (i, j) ha:
        # +1 in (i,i) e (j,j), -1 in (i,j) e (j,i)
        # Moltiplichiamo per il peso assoluto se i pesi non sono unitari
        weight = abs(L[i, j])
        edge_l = np.zeros((n, n))
        edge_l[i, i] = weight
        edge_l[j, j] = weight
        edge_l[i, j] = -weight
        edge_l[j, i] = -weight

        contributions.append(edge_l)

  return np.array(contributions)




## Calcolo autovalori e autovettori

In [ ]:
def get_fiedler_properties(L):
    """
    Calcola il secondo autovalore più piccolo (lambda2) e il relativo
    autovettore (v2) della matrice Laplaciana L.
    """
    # Calcoliamo autovalori e autovettori.
    # Poiché L è simmetrica, eigh garantisce risultati reali.
    eigenvalues, eigenvectors = linalg.eigh(L)

    # In una Laplaciana, il primo autovalore (indice 0) è sempre 0.
    # Il secondo autovalore più piccolo è all'indice 1.
    lambda2 = eigenvalues[1]
    v2 = eigenvectors[:, 1]

    return lambda2, v2


def getFiedlerVector(l):
  eigenvalues, eigenvectors = linalg.eigh(l)
  v2 = eigenvectors[:, 1]
  return v2




def printConditionNumber(matrix):
  eigenvalues, eigenvectors = linalg.eigh(matrix)

  lambda2 = eigenvalues[1]
  lambdaN = eigenvalues[-1]

  print(f"print rapporto autovalore maggiore e minore non nullo: {lambdaN/lambda2}")


def printNotmalizedEigenvalues(matrix):

  eigenvalues, eigenvectors = linalg.eigh(matrix)

  max_deg = get_max_degree_from_laplacian(matrix)
  print("Rapporto autovalori / grado massimo:")
  for eigenvalue in eigenvalues:
    print(eigenvalue/(2*max_deg))


# Passiamo al Quantum

## Codifica dell'autovettore v2 in uno stato con log2(dimensione di v2) qbit

In [ ]:
def encode_fiedler_vector(v2):
    # Creiamo un circuito con il numero di qubit necessari (log2 della lunghezza del vettore)

    num_qubits = int(np.log2(len(v2)))
    qc = QuantumCircuit(num_qubits)

    # Inizializziamo il circuito con le ampiezze del vettore v2
    qc.initialize(v2, range(num_qubits))

    return qc



## Creazione della matrice unitaria e^iHt e di e^iHe(t/n)

In [ ]:


def create_laplacian_unitary(L, t):
    """
    Crea un operatore unitario U = exp(-i * L * t) basato sulla matrice Laplaciana.
    """
    # Calcolo dell'esponenziale di matrice: U = exp(i * L * t)
    # Usiamo 1j per l'unità immaginaria
    U_matrix = expm(1j * L * t)

    # Creazione del gate unitario di Qiskit
    u_gate = UnitaryGate(U_matrix, label=f"exp(iLt)")


    return U_matrix



def compute_two_level_unitary(M,t, trotterSteps ):
    """
    Calcola exp(i*M*t) per una matrice che soddisfa M^2 = 2M.
    """
    i, j = get_non_trivial_changed_states(M)
    tau = t/trotterSteps

    # Specifichiamo dtype=complex per evitare il casting a real
    identity = np.eye(M.shape[0], dtype=complex)

    diag = np.exp(1j * tau) * np.cos(tau)
    non_diag = np.exp(1j * tau) * (-1j) * np.sin(tau)

    identity[i, i] = diag
    identity[j, j] = diag

    identity[i, j] = non_diag
    identity[j, i] = non_diag

    return identity


# Simulazione Hamiltoniana


## Trasformazione da matrici e^iHe(t/n) ad unitarie

In [ ]:

def edgeContributionsToUnitaryMatrixList(edge_contributions, time , trotterSteps):

  unitaries = []

  for edge_contribution in edge_contributions:

    unitaryExponentiated = compute_two_level_unitary(edge_contribution , time , trotterSteps )
    unitaries.append(unitaryExponentiated)

  return unitaries



## Creazione di un circuito che prende in input una lista di unitarie e le applica in sequenza per n volte

In [ ]:
def create_sequential_circuit(initial_state, unitary_matrices, repetitions):
    """
    Crea un circuito che inizializza lo stato e applica una sequenza di unitarie n volte.
    """
    num_qubits = int(np.log2(len(initial_state)))
    qc = QuantumCircuit(num_qubits)

    # 1. Inizializzazione dello stato
    qc.initialize(initial_state, range(num_qubits))
    qc.barrier() # Separatore visivo

    # 2. Applicazione della sequenza n volte
    for i in range(repetitions):
        for idx, U_mat in enumerate(unitary_matrices):
            # Trasformiamo la matrice in un gate
            u_gate = UnitaryGate(U_mat, label=f"U_{idx}")
            qc.append(u_gate, range(num_qubits))
        qc.barrier() # Barriera tra ogni ciclo completo

    return qc



## Creazione di un semplice circuito che applica l'unitaria U allo stato in input


In [ ]:
def run_simple_unitary_evolution(initial_vector, unitary_matrix):
    # 1. Calcolo del numero di qubit necessari
    num_qubits = int(np.log2(len(initial_vector)))

    # 2. Creazione del circuito
    qc = QuantumCircuit(num_qubits)

    # Inizializzazione dello stato
    qc.initialize(initial_vector, range(num_qubits))

    # Applicazione dell'unitaria
    u_gate = UnitaryGate(unitary_matrix, label="U")
    qc.append(u_gate, range(num_qubits))

    # 3. Calcolo dello stato finale
    finalStateVector = Statevector.from_instruction(qc)



    return finalStateVector




##Funzione che applica la sequenza di unitarie e^iHet/n n volte                    ( HAMILTONIAN SIMULATION )

In [ ]:
def HamiltonianSimulationByTrotterization(laplacian, initialState ,time , trotterSteps):
  # Creazione dei contributi He
  edgesContribution = get_edge_contributions(laplacian)

  #Traduzione dei contributi He in matrici unitarie
  unitaries = edgeContributionsToUnitaryMatrixList(edgesContribution,time, trotterSteps)

  #Creazione di un circuito sequenziale che prende in inpute le unitarie e le esegue in sequenza tante volte quanti sono i trotter step
  hamiltonianSimulationByTrotterizationCircuit = create_sequential_circuit(initialState, unitaries, repetitions=trotterSteps)

  #Applicazione della simulazione hamiltoniana al nostro stato |v2>
  stateAfterHamiltonianSimulation = Statevector.from_instruction(hamiltonianSimulationByTrotterizationCircuit)


  return stateAfterHamiltonianSimulation

### Confronto fra Stato simulato e stato esatto

In [ ]:

def compareHamiltonianAndExactSimulation(laplacian, initialState, time , trotterSteps, showStates=True):



  simulatedState = HamiltonianSimulationByTrotterization(laplacian, initialState ,time , trotterSteps)

  exactState = run_simple_unitary_evolution(initialState, create_laplacian_unitary(laplacian, time))
  if(showStates):
    print("Stato simulato:")
    display(simulatedState.draw('latex'))

    print("Stato esatto:")
    display(exactState.draw('latex'))

  fid = state_fidelity(exactState, simulatedState)

  print(f"\n--- Risultati Confronto (t={time}, steps={trotterSteps}) ---")
  print(f"Fedeltà tra stato esatto e stato simulato: {fid:.6f}")



# Test

### Test laplaciana sparsa

In [ ]:
l8 = generateSparseGraphWithNNodes(8)
v2 = getFiedlerVector(l8)
time = (np.pi/get_max_degree_from_laplacian(l8))

N = l8.shape[0]
trotterSteps = int(N/math.log2(N))*4 # N/logN
numPotenze = 4


for i in range (0,numPotenze):

  compareHamiltonianAndExactSimulation(l8, v2, time * (1+i), trotterSteps, showStates=False)

  trotterSteps = int(trotterSteps * 2)

Grafo generato: N=8, Grado Max=3 (Limite=3), Connesso=True

--- Risultati Confronto (t=1.0471975511965976, steps=8) ---
Fedeltà tra stato esatto e stato simulato: 0.994880

--- Risultati Confronto (t=2.0943951023931953, steps=16) ---
Fedeltà tra stato esatto e stato simulato: 0.993687

--- Risultati Confronto (t=3.141592653589793, steps=32) ---
Fedeltà tra stato esatto e stato simulato: 0.996902

--- Risultati Confronto (t=4.1887902047863905, steps=64) ---
Fedeltà tra stato esatto e stato simulato: 0.998593


## Investigazione su epsilon

In [ ]:
def runHamiltonianAndExactSimulationOnMultipleSizedLaplacian(dimensioneMassimaLaplaciana, epsilon= 0.1, potenzaMassima= None):

  logDimensione = math.log2(dimensioneMassimaLaplaciana)



  for i in range (1, int(logDimensione)+1):
    if(2 == 2**i ): continue
    laplaciana = generateSparseGraphWithNNodes(2**i)

    N = laplaciana.shape[0]


    if potenzaMassima == None:
      potenzaMassima = int(math.log2(math.sqrt(N)* math.log2(N))) # log( radice ( N) * log(N))


    trotterSteps = int(N/(math.log2(N)* epsilon)) # (N/logN*epsilon)

    if(trotterSteps > 100):continue

    print(f"Per la laplaciana con dimensione {N}, calcoliamo {potenzaMassima} potenze di U con {trotterSteps} passi di trotter inziali")

    time = (np.pi/get_max_degree_from_laplacian(laplaciana))

    v2 = getFiedlerVector(laplaciana)

    for i in range (0,potenzaMassima):

      compareHamiltonianAndExactSimulation(laplaciana, v2, time * (2**i), trotterSteps, showStates=False)

      trotterSteps = int(trotterSteps * 2)


    print("--------------------------------------")
    print("--------------------------------------")
    print("--------------------------------------")









In [ ]:
runHamiltonianAndExactSimulationOnMultipleSizedLaplacian(8, epsilon=0.1, potenzaMassima=10)

Grafo generato: N=4, Grado Max=2 (Limite=2), Connesso=True
Per la laplaciana con dimensione 4, calcoliamo 10 potenze di U con 20 passi di trotter inziali

--- Risultati Confronto (t=1.5707963267948966, steps=20) ---
Fedeltà tra stato esatto e stato simulato: 0.987726

--- Risultati Confronto (t=3.141592653589793, steps=40) ---
Fedeltà tra stato esatto e stato simulato: 1.000000

--- Risultati Confronto (t=6.283185307179586, steps=80) ---
Fedeltà tra stato esatto e stato simulato: 1.000000

--- Risultati Confronto (t=12.566370614359172, steps=160) ---
Fedeltà tra stato esatto e stato simulato: 1.000000

--- Risultati Confronto (t=25.132741228718345, steps=320) ---
Fedeltà tra stato esatto e stato simulato: 1.000000

--- Risultati Confronto (t=50.26548245743669, steps=640) ---
Fedeltà tra stato esatto e stato simulato: 1.000000

--- Risultati Confronto (t=100.53096491487338, steps=1280) ---
Fedeltà tra stato esatto e stato simulato: 1.000000

--- Risultati Confronto (t=201.06192982974676